# Lab 18: The grounded answering service

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [1]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

repository: /Users/danysigha/Desktop/LanternCS2 | LLM mode: replay


In [2]:
import inspect
print(inspect.getsource(schemas.Number)); print(inspect.getsource(schemas.Claim))
LARGE = llm.P["llm"]["models"]["large"]
FACTS = sd.seller_facts()
blocks = {r["block_id"]: r for r in sd.records("deal_a") if r["doc_id"] == "DR-deal_a-001"}   # block ids are unique only within a document
extracted = {b: llm.structured(schemas.MgmtClaims, prompts.claims_messages(blocks[b]["text"]), LARGE) for b in sd.CLAIM_BLOCKS}
CLAIMS = sd.claim_rows(extracted)

class Number(BaseModel):
    value: float; unit: str
    fact_id: Optional[str] = None    # tiers 1-3
    calc_id: Optional[str] = None    # derived
    claim_id: Optional[str] = None   # tiers 4-5
    @model_validator(mode="after")
    def one_reference(self):
        refs = (self.fact_id, self.calc_id, self.claim_id)
        if sum(r is not None for r in refs) != 1:
            raise ValueError("exactly one reference")
        return self

class Claim(BaseModel):
    text: str
    numbers: list[Number]
    citations: list[Citation]
    tier_label: Literal["filed", "seller", "management claim", "web"]
    @model_validator(mode="after")
    def claim_ids_need_a_label(self):
        ok = ("management claim", "web")
        uses = any(n.claim_id for n in self.numbers)
        if uses and self.tier_label not in ok:
            raise ValueError("claim_id needs a claim label")
        return self



In [3]:
# From the slide: The numeric-claim linter
import math, re
NUM = re.compile(r"""(?P<neg>\()?\$?\s*
  (?P<num>\d{1,3}(?:,\d{3})+(?:\.\d+)?|\d+(?:\.\d+)?)
  \s*(?P<unit>%|x\b|bn\b|mm\b|m\b|million\b|k\b)?
  (?(neg)\))""", re.X | re.I)
YEAR = re.compile(r"^(19|20)\d{2}$")
SCALE = {"bn": 1e9, "mm": 1e6, "m": 1e6, "million": 1e6,
         "k": 1e3}

def numbers_in(text):
    for m in NUM.finditer(text):
        raw, unit = m["num"], (m["unit"] or "").lower()
        a, b = m.start("num"), m.end("num")
        if (YEAR.match(raw) and not unit
                or text[a-1:a].isalpha()   # Q2, FY24
                or re.match(r"-[A-Za-z]", text[b:b+2])
                or re.search(r"(item|note)\s*$",
                             text[max(0, a-6):a], re.I)):
            continue               # 10-K, 8-K, Item 7
        v = float(raw.replace(",", ""))
        v *= SCALE.get(unit, 1)
        yield -v if m["neg"] else v

def lint(text, allowed, tol=0.005):     # [] == pass
    ok = lambda v: any(math.isclose(v, a, rel_tol=tol)
                       for a in allowed)
    return [v for v in numbers_in(text) if not ok(v)]

In [4]:
fact_by_id = {f["fact_id"]: f for f in FACTS}; claim_by_id = {c["claim_id"]: c for c in CLAIMS}
def allowed_for(claim):
    vals = []
    for n in claim.numbers:
        if n.fact_id: f = fact_by_id[n.fact_id]; v, pct = f["value"], f["unit"] == "pct"
        elif n.claim_id: c = claim_by_id[n.claim_id]; v, pct = c["value_abs"], c["unit"] == "pct"
        else: continue                          # calc_id: add your calculator outputs here (Labs 19-20)
        vals += [v, v * 100] if pct else [v]    # the linter reads "24.6%" as 24.6
    return vals
def answer(task_id):
    t = sd.dev_tasks()[task_id]; q = t["turns"][0]["content"]
    msgs = prompts.answer_messages(q, FACTS, CLAIMS, sd.evidence_for(task_id))
    ans = llm.structured(schemas.Answer, msgs, LARGE)
    bad = [v for c in ans.claims for v in lint(c.text, allowed_for(c))]
    if bad:                                         # regenerate once with the linter's feedback
        print(f"  linter rejected {bad}; regenerating")
        ans = llm.structured(schemas.Answer, prompts.regenerate_messages(msgs, ans.model_dump_json(), bad), LARGE)
        keep = [c for c in ans.claims if not lint(c.text, allowed_for(c))]
        if len(keep) < len(ans.claims): ans = schemas.Answer(claims=keep, abstained=not keep, abstain_reason="partially abstained")
    return q, ans
Path("outputs/qa").mkdir(parents=True, exist_ok=True)
with open("outputs/qa/dev.jsonl", "w") as fh:
    for tid in sd.TASK_EVIDENCE:
        q, ans = answer(tid); print(f"Q: {q}")
        print("  Not disclosed." if ans.abstained and not ans.claims else "", ans.abstain_reason or "")
        for c in ans.claims: print(f"  [{c.tier_label}] {c.text}")
        fh.write(json.dumps({"task_id": tid, "answer": ans.model_dump()}) + "\n")

Q: What was Wren's FY2025 revenue?
 
  [seller] Wren's FY2025 revenue was $41.2 million, per seller financials.
  linter rejected [118.0]; regenerating
Q: What was Wren's net revenue retention in FY2025, per the seller's KPIs?
 
  [seller] Net revenue retention was 109% in FY2025, per seller financials.
  [management claim] Per the CIM (unverified), net revenue retention was 118%, which management calls its best quarter annualized.
Q: What is Wren's FY2026 budgeted EBITDA?
  Not disclosed. Not disclosed: the data room does not include an FY2026 budget.


The service from the slide, written to `services/qa/app.py`. Run it with `uvicorn services.qa.app:app` from the repository root.

In [5]:
service = """import sys, json
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2]))
from src.schemas import Answer
from src.llm.client import MODE
from src import sample_data as sd
facts = {f["fact_id"]: f for f in sd.seller_facts()}
def grounded_answer(question, scope):
    return Answer(claims=[], abstained=True, abstain_reason="Not disclosed: wire in the notebook-18 pipeline here.")
""" + 'from fastapi import FastAPI, HTTPException\nfrom pydantic import BaseModel\n\napp = FastAPI(title="LANTERN QA")\nALLOWED_SCOPES = {"deal_a", "deal_b", "kestrel", "public"}\n\nclass Ask(BaseModel):\n    question: str\n    scope: str                   # required, no default\n\n@app.post("/answer", response_model=Answer)\ndef answer(req: Ask) -> Answer:\n    if req.scope not in ALLOWED_SCOPES:\n        raise HTTPException(400, "unknown scope")\n    # grounded_answer: retrieve, generate, lint, cite\n    return grounded_answer(req.question, req.scope)\n@app.get("/facts/{fact_id}")        # and /calc/{calc_id}\ndef fact(fact_id: str):\n    if fact_id not in facts:\n        raise HTTPException(404, "unknown fact")\n    return facts[fact_id]\n@app.get("/health")\ndef health():\n    return {"ok": True, "llm_mode": MODE}'
Path("services/qa").mkdir(parents=True, exist_ok=True); Path("services/qa/app.py").write_text(service)
try:
    from fastapi.testclient import TestClient
    import importlib.util
    spec = importlib.util.spec_from_file_location("qa_app", "services/qa/app.py"); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
    cl = TestClient(m.app)
    print(cl.get("/health").json(), cl.post("/answer", json={"question": "q", "scope": "nowhere"}).status_code)
except ImportError:
    print("pip install fastapi httpx to test the service here")

{'ok': True, 'llm_mode': 'replay'} 400


**Exercise 18.** Write five numbers the linter must accept and five it must reject (include a year, a form name and a period label). Does it pass all ten?

**Exercise 18 result.** These are the twelve starter cases from the handout. The final two `BAD` results are expected false alarms for a page reference and a date.

In [6]:
cases = [("$52.3 million", [52.3e6]), ("($1,200)", [-1200.0]),
         ("14.2x", [14.2]), ("24.6%", [24.6]), ("$1.2bn", [1.2e9]),
         ("FY2025", []), ("Item 7", []), ("Q2", []), ("10-K", []),
         ("in 2019", []),
         ("on page 41", []), ("March 31, 2026", [])]
for text, want in cases:
    got = list(numbers_in(text))
    print("ok " if got == want else "BAD", repr(text), got)

ok  '$52.3 million' [52300000.0]
ok  '($1,200)' [-1200.0]
ok  '14.2x' [14.2]
ok  '24.6%' [24.6]
ok  '$1.2bn' [1200000000.0]
ok  'FY2025' []
ok  'Item 7' []
ok  'Q2' []
ok  '10-K' []
ok  'in 2019' []
BAD 'on page 41' [41.0]
BAD 'March 31, 2026' [31.0]


**Offline "make it lie" rehearsal.** Run in a temporary notebook copy with `bad = []`; only `dev-w-02` is called.

- Draft: `[seller] Net revenue retention was 118% in FY2025.` It cites `F-69cbc9e5f9`, whose value is `1.09`.
- Linted answer: `[seller] Net revenue retention was 109% in FY2025, per seller financials.` and `[management claim] Per the CIM (unverified), net revenue retention was 118%, which management calls its best quarter annualized.`

**Service smoke test.** `/health` returned `{"ok": true, "llm_mode": "replay"}`. `/answer` with `deal_a` returned the documented not-disclosed stub, an unknown scope returned `400`, and an omitted scope returned validation status `422`.